In [85]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import dagshub
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, classification_report)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

from imblearn.over_sampling import SMOTENC, SMOTE
from imblearn.under_sampling import TomekLinks

from dotenv import load_dotenv
load_dotenv('../.env', override=True)

import os
DAGSHUB_USERNAME  = os.environ['DAGSHUB_USERNAME']
DAGSHUB_REPO_NAME = os.environ['DAGSHUB_REPO_NAME']
DAGSHUB_TOKEN     = os.environ['DAGSHUB_TOKEN']
MLFLOW_EXPERIMENT = os.getenv('MLFLOW_EXPERIMENT', 'wine-quality')
REGISTERED_MODEL  = os.getenv('MLFLOW_MODEL_NAME',  'wine-quality')

print("✅ Imports OK")


✅ Imports OK


## Carregando Dataset

In [86]:
df = pd.read_csv("../data/wine_quality_merged.csv")
# df = pd.read_csv("../data/winequality-red.csv")
print('Classes :', sorted(df['quality'].unique()))
print(df['quality'].value_counts().sort_index().to_string())   


Classes : [np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
quality
3      30
4     216
5    2138
6    2836
7    1079
8     193
9       5


## 1. EDA

## 2. Fusão de Classes
| Label | Classe original | Descrição |
|-------|-----------------|-----------|
| **0** | 3, 4, 5 | Ruim |
| **1** | 6 | Médio |
| **2** | 7, 8, 9 | Bom |


In [87]:
df_merged = df.copy()
df_merged["quality"] = df["quality"].apply(lambda x: "0" if x <= 5 else ("1" if x == 6 else "2"))
df_merged['quality'].replace("0",0, inplace=True)
df_merged['quality'].replace("1",1, inplace=True)
df_merged['quality'].replace("2",2, inplace=True)

# separar X e y
X = df_merged.drop(columns=['quality'])
y_orig = df_merged['quality']
print('Classes após fusão:', sorted(df_merged['quality'].unique()))
print(df_merged['quality'].value_counts().sort_index().to_string())



Classes após fusão: [np.int64(0), np.int64(1), np.int64(2)]
quality
0    2384
1    2836
2    1277


## 3. Split Treino / Validação / Teste  (60 / 20 / 20)

In [88]:
# 1ª divisão: 80% temp + 20% teste
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_orig, test_size=0.20, random_state=42, stratify=y_orig
)
# 2ª divisão: 75% treino + 25% validação  →  60% / 20% do total
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

total = len(X)
print(f"Treino:    {len(X_train):>5} amostras  ({len(X_train)/total*100:.1f}%)")
print(f"Validação: {len(X_val):>5} amostras  ({len(X_val)/total*100:.1f}%)")
print(f"Teste:     {len(X_test):>5} amostras  ({len(X_test)/total*100:.1f}%)")
print("\nDistribuição no treino:")
print(y_train.value_counts().sort_index())


Treino:     3897 amostras  (60.0%)
Validação:  1300 amostras  (20.0%)
Teste:      1300 amostras  (20.0%)

Distribuição no treino:
quality
0    1430
1    1701
2     766
Name: count, dtype: int64


### Models

In [89]:
# ── Identificar colunas ──────────────────────────────────────
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
cat_idx  = [X.columns.get_loc(c) for c in cat_cols]

print(f"Numéricas  ({len(num_cols)}): {num_cols}")
print(f"Categóricas({len(cat_cols)}): {cat_cols}")

# ── Fábrica de pré-processador ───────────────────────────────
def make_preprocessor():
    steps = [("num", StandardScaler(), num_cols)]
    if cat_cols:
        steps.append(("cat",
                       OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                       cat_cols))
    return ColumnTransformer(steps)

# ── Fábrica de modelos (instâncias frescas) ───────────────────
def get_models():
    return {
        "logistic_regression": LogisticRegression(max_iter=1000, random_state=42),
        "random_forest":       RandomForestClassifier(n_estimators=100, random_state=42),
        "knn":                 KNeighborsClassifier(),
        "svm":                 SVC(random_state=42),
        "xgboost":             XGBClassifier(eval_metric="mlogloss", random_state=42),
        "mlp":                 MLPClassifier(hidden_layer_sizes=(200, 100),
                                             max_iter=2000, random_state=42),
    }

# ── Cálculo de métricas ───────────────────────────────────────
def compute_metrics(y_true, y_pred, prefix=""):
    return {
        f"{prefix}accuracy":  round(accuracy_score(y_true, y_pred), 4),
        f"{prefix}precision": round(precision_score(y_true, y_pred,
                                                    average="weighted",
                                                    zero_division=0), 4),
        f"{prefix}recall":    round(recall_score(y_true, y_pred,
                                                 average="weighted",
                                                 zero_division=0), 4),
        f"{prefix}f1":        round(f1_score(y_true, y_pred,
                                             average="weighted",
                                             zero_division=0), 4),
    }

print("✅ Utilitários definidos")


Numéricas  (11): ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']
Categóricas(1): ['type']
✅ Utilitários definidos


## 5. Configuração DagHub / MLflow

In [90]:
dagshub.init(repo_owner=DAGSHUB_USERNAME, repo_name=DAGSHUB_REPO_NAME, mlflow=True)
mlflow.set_experiment("wine_quality_classification")

print("✅ MLflow tracking URI:", mlflow.get_tracking_uri())


Initialized MLflow to track repo "frpbotero/wine-quality"

Repository frpbotero/wine-quality initialized!

✅ MLflow tracking URI: https://dagshub.com/frpbotero/wine-quality.mlflow


---
## Estratégia 1 — SMOTE (Over-sampling)
Aplica **SMOTENC** (suporte a features categóricas) no conjunto de treino para  
balancear as classes.  
Cada modelo é embrulhado num `Pipeline` com pré-processador próprio.


### 6. Aplicar SMOTE ao conjunto de treino

In [91]:
if cat_idx:
    sampler = SMOTENC(categorical_features=cat_idx, random_state=42)
    print("Usando SMOTENC (há features categóricas)")
else:
    sampler = SMOTE(random_state=42)
    print("Usando SMOTE (apenas features numéricas)")

X_train_smote, y_train_smote = sampler.fit_resample(X_train, y_train)

# Manter como DataFrame para compatibilidade com o Pipeline
X_train_smote = pd.DataFrame(X_train_smote, columns=X_train.columns)
y_train_smote = np.array(y_train_smote, dtype=int)

print("\nDistribuição após SMOTE:")
print(pd.Series(y_train_smote).value_counts().sort_index())
print(f"\nTotal de amostras: {len(X_train_smote)}")


Usando SMOTENC (há features categóricas)

Distribuição após SMOTE:
0    1701
1    1701
2    1701
Name: count, dtype: int64

Total de amostras: 5103


### 7. Treinar, validar, testar e registrar — SMOTE

In [92]:
STRATEGY = "SMOTE"
results_smote = {}

for model_name, model in get_models().items():
    print(f"\n{'='*60}")
    print(f"  Estratégia: {STRATEGY}  |  Modelo: {model_name}")
    print(f"{'='*60}")

    # ── Pipeline: pré-processador + classificador ─────────────
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("classifier",   model),
    ])

    # ── Treino ────────────────────────────────────────────────
    pipeline.fit(X_train_smote, y_train_smote)

    # ── Validação ─────────────────────────────────────────────
    y_val_pred  = pipeline.predict(X_val)
    val_metrics = compute_metrics(y_val, y_val_pred, prefix="val_")

    # ── Teste ─────────────────────────────────────────────────
    y_test_pred  = pipeline.predict(X_test)
    test_metrics = compute_metrics(y_test, y_test_pred, prefix="test_")

    all_metrics = {**val_metrics, **test_metrics}
    results_smote[model_name] = all_metrics

    # ── Imprimir resultados ───────────────────────────────────
    print(f"  Validação → Acc:{val_metrics['val_accuracy']:.4f}  "
          f"P:{val_metrics['val_precision']:.4f}  "
          f"R:{val_metrics['val_recall']:.4f}  "
          f"F1:{val_metrics['val_f1']:.4f}")
    print(f"  Teste     → Acc:{test_metrics['test_accuracy']:.4f}  "
          f"P:{test_metrics['test_precision']:.4f}  "
          f"R:{test_metrics['test_recall']:.4f}  "
          f"F1:{test_metrics['test_f1']:.4f}")
    print("\n" + classification_report(y_test, y_test_pred,
                                        target_names=["Ruim(0)", "Médio(1)", "Bom(2)"]))

    # ── Log no DagHub / MLflow ────────────────────────────────
    with mlflow.start_run(run_name=f"{STRATEGY}_{model_name}"):
        mlflow.log_param("strategy",      STRATEGY)
        mlflow.log_param("model",         model_name)
        mlflow.log_param("train_samples", int(len(X_train_smote)))
        mlflow.log_param("val_samples",   int(len(X_val)))
        mlflow.log_param("test_samples",  int(len(X_test)))
        mlflow.log_metrics(all_metrics)
        mlflow.sklearn.log_model(
            pipeline,
            artifact_path="model",
            registered_model_name=f"wine_{STRATEGY}_{model_name}",
        )
    print(f"  ✅ Registrado → wine_{STRATEGY}_{model_name}")

print("\n🎉 SMOTE — todos os modelos registrados no DagHub!")



  Estratégia: SMOTE  |  Modelo: logistic_regression
  Validação → Acc:0.5531  P:0.5714  R:0.5531  F1:0.5438
  Teste     → Acc:0.5431  P:0.5593  R:0.5431  F1:0.5348

              precision    recall  f1-score   support

     Ruim(0)       0.63      0.68      0.65       477
    Médio(1)       0.56      0.36      0.44       567
      Bom(2)       0.42      0.70      0.53       256

    accuracy                           0.54      1300
   macro avg       0.54      0.58      0.54      1300
weighted avg       0.56      0.54      0.53      1300



2026/05/03 14:34:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 14:34:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'wine_SMOTE_logistic_regression' already exists. Creating a new version of this model...
2026/05/03 14:34:50 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: wine_SMOTE_logistic_regression, version 3
Created version '3' of model 'wine_SMOTE_logistic_regression'.


🏃 View run SMOTE_logistic_regression at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1/runs/5370548708f04496b1b06ade24c8c87a
🧪 View experiment at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1
  ✅ Registrado → wine_SMOTE_logistic_regression

  Estratégia: SMOTE  |  Modelo: random_forest
  Validação → Acc:0.6877  P:0.6893  R:0.6877  F1:0.6866
  Teste     → Acc:0.6923  P:0.6945  R:0.6923  F1:0.6913

              precision    recall  f1-score   support

     Ruim(0)       0.75      0.78      0.76       477
    Médio(1)       0.69      0.61      0.65       567
      Bom(2)       0.61      0.72      0.66       256

    accuracy                           0.69      1300
   macro avg       0.68      0.70      0.69      1300
weighted avg       0.69      0.69      0.69      1300



2026/05/03 14:35:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run SMOTE_random_forest at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1/runs/0f5a2412593f4158b6712eeb19c278e2
🧪 View experiment at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1


KeyboardInterrupt: 

---
## Estratégia 2 — Tomek Links (Under-sampling)
**TomekLinks** requer features numéricas, portanto o pré-processamento  
é aplicado primeiro e compartilhado entre todos os modelos.


### 8. Pré-processar e aplicar Tomek Links

In [ ]:
# ── Pré-processamento ────────────────────────────────────────
preprocessor_tomek = make_preprocessor()
X_train_trans = preprocessor_tomek.fit_transform(X_train)
X_val_trans   = preprocessor_tomek.transform(X_val)
X_test_trans  = preprocessor_tomek.transform(X_test)

y_train_int = y_train.values.astype(int)
y_val_int   = y_val.values.astype(int)
y_test_int  = y_test.values.astype(int)

# ── Tomek Links ───────────────────────────────────────────────
tomek = TomekLinks()
X_train_tomek, y_train_tomek = tomek.fit_resample(X_train_trans, y_train_int)

print("Distribuição após Tomek Links:")
print(pd.Series(y_train_tomek).value_counts().sort_index())
print(f"\nAmostras removidas: {len(X_train_trans) - len(X_train_tomek)}")


### 9. Treinar, validar, testar e registrar — Tomek Links

In [ ]:
STRATEGY = "Tomek"
results_tomek = {}

for model_name, model in get_models().items():
    print(f"\n{'='*60}")
    print(f"  Estratégia: {STRATEGY}  |  Modelo: {model_name}")
    print(f"{'='*60}")

    # ── Treino direto nos dados já transformados ───────────────
    model.fit(X_train_tomek, y_train_tomek)

    # ── Validação ─────────────────────────────────────────────
    y_val_pred  = model.predict(X_val_trans)
    val_metrics = compute_metrics(y_val_int, y_val_pred, prefix="val_")

    # ── Teste ─────────────────────────────────────────────────
    y_test_pred  = model.predict(X_test_trans)
    test_metrics = compute_metrics(y_test_int, y_test_pred, prefix="test_")

    all_metrics = {**val_metrics, **test_metrics}
    results_tomek[model_name] = all_metrics

    # ── Imprimir resultados ───────────────────────────────────
    print(f"  Validação → Acc:{val_metrics['val_accuracy']:.4f}  "
          f"P:{val_metrics['val_precision']:.4f}  "
          f"R:{val_metrics['val_recall']:.4f}  "
          f"F1:{val_metrics['val_f1']:.4f}")
    print(f"  Teste     → Acc:{test_metrics['test_accuracy']:.4f}  "
          f"P:{test_metrics['test_precision']:.4f}  "
          f"R:{test_metrics['test_recall']:.4f}  "
          f"F1:{test_metrics['test_f1']:.4f}")
    print("\n" + classification_report(y_test_int, y_test_pred,
                                        target_names=["Ruim(0)", "Médio(1)", "Bom(2)"]))

    # ── Log no DagHub / MLflow ────────────────────────────────
    with mlflow.start_run(run_name=f"{STRATEGY}_{model_name}"):
        mlflow.log_param("strategy",      STRATEGY)
        mlflow.log_param("model",         model_name)
        mlflow.log_param("train_samples", int(len(X_train_tomek)))
        mlflow.log_param("val_samples",   int(len(X_val_trans)))
        mlflow.log_param("test_samples",  int(len(X_test_trans)))
        mlflow.log_metrics(all_metrics)

        # Modelo treinado
        mlflow.sklearn.log_model(
            model,
            artifact_path="model",
            registered_model_name=f"wine_{STRATEGY}_{model_name}",
        )
        # Pré-processador compartilhado (necessário para inferência futura)
        mlflow.sklearn.log_model(
            preprocessor_tomek,
            artifact_path="preprocessor",
        )
    print(f"  ✅ Registrado → wine_{STRATEGY}_{model_name}")

print("\n🎉 Tomek — todos os modelos registrados no DagHub!")


---
## 10. Resumo Final

In [ ]:
import pandas as pd

rows = []
for model_name in get_models().keys():
    sm = results_smote.get(model_name, {})
    tk = results_tomek.get(model_name, {})
    rows.append({
        "Modelo":          model_name,
        "SMOTE  Val-F1":   sm.get("val_f1",  "-"),
        "SMOTE  Test-F1":  sm.get("test_f1", "-"),
        "SMOTE  Test-Acc": sm.get("test_accuracy", "-"),
        "Tomek  Val-F1":   tk.get("val_f1",  "-"),
        "Tomek  Test-F1":  tk.get("test_f1", "-"),
        "Tomek  Test-Acc": tk.get("test_accuracy", "-"),
    })

summary = pd.DataFrame(rows).set_index("Modelo")
print(summary.to_string())
print("\n🏆 Melhor Test-F1 SMOTE :", summary["SMOTE  Test-F1"].max(),
      "→", summary["SMOTE  Test-F1"].idxmax())
print("🏆 Melhor Test-F1 Tomek :", summary["Tomek  Test-F1"].max(),
      "→", summary["Tomek  Test-F1"].idxmax())


In [ ]:

# # salvar pipeline treinado
# with open("mlp_wine.pkl", "wb") as f:
#     pickle.dump(mlp_pipeline, f)
# print("Modelo salvo em mlp_wine.pkl")


In [ ]:
# # treinar classificador diretamente sobre os dados já transformados
# from sklearn.neural_network import MLPClassifier
# clf = MLPClassifier(hidden_layer_sizes=(200,100), max_iter=2000, random_state=42)
# clf.fit(X_train_tomek, y_train_tomek)

# # avaliar usando X_test_trans
# y_pred = clf.predict(X_test_trans)
# print("Accuracy:", accuracy_score(y_test_num, y_pred))
# print(classification_report(y_test_num, y_pred))